# 05 — Contre-vérification

Tous nos chiffres (RMSE, R² médian, % de paires EPA) sortent d'une seule fonction, `resume` dans
`src/aircalib/metriques.py`. Si elle avait un bug, tout le projet serait faux sans qu'on le voie.

**Ce qu'on fait.** On recalcule ces chiffres **sans rien importer de `src/`**, avec seulement `numpy` et `pyarrow`,
à partir des prédictions hors pli sauvegardées par `scripts/04_entrainer.py`
(`data/predictions/5_lightgbm_combine.parquet`). On écrit les formules à la main, avec des sommes, au lieu
d'utiliser Polars. Puis on compare au journal des essais `reports/runs.csv`.

**Pré-requis.** `make data` puis `make train`. Pour tester le notebook en une minute, mettre `ECHANTILLON = True`
après `make sample`.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq

ECHANTILLON = False  # True : les sorties de « make sample » (data/echantillon/)
RACINE = Path.cwd().parent
DONNEES = RACINE / "data" / "echantillon" if ECHANTILLON else RACINE / "data"
JOURNAL = DONNEES / "runs.csv" if ECHANTILLON else RACINE / "reports" / "runs.csv"
MODELE = "5_lightgbm_combine"
PERIMETRE = "échantillon de test" if ECHANTILLON else "toutes les stations"  # comme scripts/04_entrainer.py

# Les cibles EPA (Duvall et al., 2021) et nos seuils, recopiés à la main depuis le README.
R2_MIN, PENTE, ORDONNEE, RMSE_MAX = 0.70, (0.65, 1.35), (-5.0, 5.0), 7.0
HEURES_MIN_PAIRE, JOURS_MIN_PAIRE, HEURES_MIN_JOUR = 500, 30, 18

colonnes = ["capteur", "heure", "reference", "prediction", "paire_saine"]
table = pq.read_table(DONNEES / "predictions" / f"{MODELE}.parquet", columns=colonnes)
saine = table["paire_saine"].to_numpy(zero_copy_only=False)
capteur = table["capteur"].to_numpy()[saine]
heure = table["heure"].to_numpy()[saine]
x = table["reference"].to_numpy()[saine]  # la station
y = table["prediction"].to_numpy()[saine]  # le capteur corrigé
print(f"{len(x):,} heures-paires saines, {len(np.unique(capteur))} capteurs")

## 1. Les formules, avec des sommes

Pour chaque paire, l'EPA régresse la prédiction $y$ sur la référence $x$ : $y = a\,x + b$. Avec $n$ heures et les
sommes $S_x = \sum x$, $S_{xx} = \sum x^2$, $S_{xy} = \sum xy$, etc. :

$$a = \frac{S_{xy} - S_x S_y / n}{S_{xx} - S_x^2 / n}, \qquad b = \bar y - a\,\bar x, \qquad
R^2 = \frac{(S_{xy} - S_x S_y / n)^2}{(S_{xx} - S_x^2 / n)(S_{yy} - S_y^2 / n)}, \qquad
\text{RMSE} = \sqrt{\tfrac1n \sum (y - x)^2}$$

On calcule toutes les sommes d'un coup pour chaque groupe avec `np.bincount`, sans boucle sur les paires.

In [ ]:
def stats_par_groupe(groupe: np.ndarray, x: np.ndarray, y: np.ndarray) -> dict[str, np.ndarray]:
    """n, pente, ordonnée, R² et RMSE de la régression de y sur x, pour chaque valeur de `groupe`."""
    _, g = np.unique(groupe, return_inverse=True)

    def somme(v):
        return np.bincount(g, weights=v)

    n = np.bincount(g).astype(float)
    sx, sy = somme(x), somme(y)
    cxx = somme(x * x) - sx**2 / n
    cyy = somme(y * y) - sy**2 / n
    cxy = somme(x * y) - sx * sy / n
    pente = cxy / cxx
    return {
        "n": n,
        "pente": pente,
        "ordonnee": sy / n - pente * sx / n,
        "r2": cxy**2 / (cxx * cyy),
        "rmse": np.sqrt(somme((y - x) ** 2) / n),
    }


def passe_epa(s: dict[str, np.ndarray]) -> np.ndarray:
    return (
        (s["r2"] >= R2_MIN)
        & (s["pente"] >= PENTE[0])
        & (s["pente"] <= PENTE[1])
        & (s["ordonnee"] >= ORDONNEE[0])
        & (s["ordonnee"] <= ORDONNEE[1])
        & (s["rmse"] <= RMSE_MAX)
    )


# Contrôle sur un cas connu : y = 2x + 1 exactement, donc pente 2, ordonnée 1, R² = 1.
essai = stats_par_groupe(np.zeros(5), np.arange(5.0), 2 * np.arange(5.0) + 1)
assert np.isclose(essai["pente"], 2) and np.isclose(essai["ordonnee"], 1) and np.isclose(essai["r2"], 1)
print("formules vérifiées sur y = 2x + 1")

## 2. Horaire et journalier

- **Horaire :** une paire est notée si elle a au moins 500 heures.
- **Journalier :** on moyenne d'abord chaque paire par jour (UTC), en gardant les jours d'au moins 18 heures, puis on
  note les paires qui ont au moins 30 jours complets.

In [ ]:
horaire = stats_par_groupe(capteur, x, y)
notees = horaire["n"] >= HEURES_MIN_PAIRE

# Un identifiant par (capteur, jour) : le numéro du jour depuis 1970, accolé au numéro du capteur.
jour = heure.astype("datetime64[D]").astype(np.int64)
cle = capteur.astype(np.int64) * 100_000 + jour
cles, g, heures_par_jour = np.unique(cle, return_inverse=True, return_counts=True)
x_jour = np.bincount(g, weights=x) / heures_par_jour
y_jour = np.bincount(g, weights=y) / heures_par_jour
complet = heures_par_jour >= HEURES_MIN_JOUR
journalier = stats_par_groupe(cles[complet] // 100_000, x_jour[complet], y_jour[complet])
notees_j = journalier["n"] >= JOURS_MIN_PAIRE

a_la_main = {
    "rmse": float(np.sqrt(np.mean((y - x) ** 2))),
    "r2_median_par_paire": float(np.median(horaire["r2"][notees])),
    "pct_epa_horaire": float(100 * passe_epa(horaire)[notees].mean()) if notees.any() else float("nan"),
    "pct_epa_journalier": float(100 * passe_epa(journalier)[notees_j].mean())
    if notees_j.any()
    else float("nan"),
}
print(f"paires notées : {notees.sum()} en horaire, {notees_j.sum()} en journalier")
a_la_main

## 3. Comparaison avec le journal des essais

On prend la dernière ligne du modèle retenu **sur toutes les stations** dans le journal (écrite par
`scripts/04_entrainer.py`), et on compare. Les lignes « échantillon : 20 % des stations » viennent du notebook 03 et
ne correspondent pas à ces prédictions.
Le journal arrondit à 3 décimales (RMSE, R²) et à 1 décimale (pourcentages) : on tolère donc un écart d'arrondi.

In [ ]:
with open(JOURNAL, encoding="utf-8") as f:
    ligne = [r for r in csv.DictReader(f) if r["modele"] == MODELE and r["perimetre"] == PERIMETRE][-1]

tolerance = {"rmse": 0.001, "r2_median_par_paire": 0.001, "pct_epa_horaire": 0.05, "pct_epa_journalier": 0.05}
print(f"{'chiffre':<22}{'src/':>10}{'à la main':>12}{'écart':>10}")
for nom, tol in tolerance.items():
    officiel = float(ligne[nom]) if ligne[nom] else float("nan")
    ecart = abs(a_la_main[nom] - officiel)
    print(f"{nom:<22}{officiel:>10.3f}{a_la_main[nom]:>12.3f}{ecart:>10.4f}")
    assert np.isnan(officiel) and np.isnan(a_la_main[nom]) or ecart <= tol, f"écart sur {nom}"
print("\nles deux calculs concordent")

**Ce que ça veut dire :** si la cellule précédente affiche « les deux calculs concordent », les chiffres du README
et du notebook 04 ne dépendent pas d'un bug de notre fonction `resume` : deux implémentations écrites différemment
(Polars d'un côté, des sommes en numpy de l'autre) donnent le même résultat à l'arrondi près.

Ce notebook ne vérifie pas tout : il contrôle le **calcul des métriques**, pas le nettoyage ni la validation croisée
(les prédictions sont prises telles quelles).